# Europe PMC — Literature Mining and Article Metadata Analysis

**Europe PMC** is a free, open-access repository of biomedical and life-sciences literature maintained by the European Bioinformatics Institute (EMBL-EBI) and a consortium of funders. It provides programmatic access to over **40 million abstracts** and **5 million full-text articles**, integrating content from PubMed/MEDLINE, PubMed Central, preprint servers (bioRxiv, medRxiv), and patent literature.

| Data Type | Description |
|---|---|
| Article metadata | Title, authors, journal, DOI, publication date, source |
| Abstracts | Free-text structured and unstructured abstracts |
| Full-text XML | Complete open-access articles in JATS/NLM XML |
| Citations & references | Cited-by counts and structured reference lists |
| Annotations | Named entity annotations (genes, diseases, chemicals, organisms) |
| Grant information | Funding agency and grant number linkage |
| Preprints | bioRxiv / medRxiv records with version tracking |

**Reference:** Europe PMC Consortium (2023), *Nucleic Acids Research*, Europe PMC: https://europepmc.org  
**REST API:** https://www.ebi.ac.uk/europepmc/webservices/rest

In [ ]:
import requests
import time
import json
import xml.etree.ElementTree as ET
from pathlib import Path

import polars as pl

# TODO

* [x] **Ingest data**
    * [x] Connect to the Europe PMC REST API and confirm access
    * [x] Search for articles via `/search` endpoint with a biological query
    * [x] Page through results and download article metadata into a Polars DataFrame
    * [x] Fetch full-text XML for a specific article of interest
    * [x] Save data to `data/` with caching (skip download if file exists)
* [ ] **Explore and clean**
    * [ ] Summarise dataset dimensions and field coverage (nulls, dtypes)
    * [ ] Parse and normalise publication dates; analyse temporal trends
    * [ ] Examine open-access status, source breakdown, and citation counts
* [ ] **Analysis**
    * [ ] Extract and rank most-cited articles in the result set
    * [ ] Identify prolific authors and journals in the query results
    * [ ] Parse grant/funder information and summarise funding landscape
* [ ] **Visualization**
    * [ ] Plot publication volume over time (line/bar chart)
    * [ ] Visualise journal and source distributions (bar/pie charts)
    * [ ] Word-frequency analysis of titles and abstracts (word cloud or bar plot)
* [ ] **Statistical analysis**
    * [ ] Model publication growth rate (exponential fit) and forecast future output
    * [ ] Compare citation distributions across open-access vs. restricted articles
    * [ ] Discuss multiple hypothesis correction considerations for text-mining analyses

## 1. Ingest Data

### 1.1 Connect to the Europe PMC REST API and Confirm Access

In [ ]:
EPMC_BASE = "https://www.ebi.ac.uk/europepmc/webservices/rest"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


def epmc_get(endpoint: str, params: dict = None) -> requests.Response:
    """
    Send a GET request to the Europe PMC REST API with a polite delay.

    Parameters
    ----------
    endpoint : str
        API path relative to EPMC_BASE (e.g. "search" or "article/MED/12345678").
    params : dict, optional
        Query parameters to include in the request.

    Returns
    -------
    requests.Response
        Raw response; caller is responsible for parsing (JSON or XML).

    Notes
    -----
    A 0.5-second sleep is injected after every call to respect EBI's
    fair-use guidance and avoid rate-limiting.
    """
    url = f"{EPMC_BASE}/{endpoint}"
    resp = requests.get(url, params=params or {}, timeout=60)
    resp.raise_for_status()
    time.sleep(0.5)  # polite inter-call delay
    return resp


# ── Connectivity check: fetch a single well-known article ────────────────────
# PMC:PMC1064873 is the original Europe PMC description paper.
# We use the /article endpoint to confirm the API is reachable.
test_resp = epmc_get("article/PMC/PMC1064873", params={"format": "json"})
test_article = test_resp.json()

title  = test_article.get("title", "N/A")
source = test_article.get("source", "N/A")
pub_year = test_article.get("pubYear", "N/A")

print(f"API connectivity: OK")
print(f"Title   : {title}")
print(f"Source  : {source}")
print(f"PubYear : {pub_year}")

### 1.2 Search for Articles and Page Through Results

In [ ]:
SEARCH_QUERY = "CRISPR genome editing"  # biologically rich topic with plenty of literature
SEARCH_CACHE = DATA_DIR / "epmc_crispr.json"
PAGE_SIZE = 1000  # Europe PMC supports up to 1000 results per page


def fetch_epmc_articles(
    query: str,
    cache_path: Path,
    page_size: int = PAGE_SIZE,
    max_pages: int = 5,
) -> list[dict]:
    """
    Search Europe PMC and page through results, caching to disk.

    Europe PMC uses cursor-based pagination: each response contains a
    ``nextCursorMark`` token that must be forwarded in the subsequent
    request. Iteration stops when the token is absent or unchanged, or
    when ``max_pages`` is reached to keep the notebook demo manageable.

    Parameters
    ----------
    query : str
        Lucene-style query string, e.g. ``"CRISPR genome editing"``.
    cache_path : Path
        File path for on-disk JSON cache; subsequent runs load from here.
    page_size : int
        Number of results per API call (max 1000 for Europe PMC).
    max_pages : int
        Upper bound on the number of pages to fetch; prevents inadvertent
        retrieval of millions of records during a demo.

    Returns
    -------
    list[dict]
        Flat list of article result dicts from the Europe PMC ``resultList``.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    all_articles: list[dict] = []
    cursor_mark = "*"  # initial cursor; Europe PMC uses deep-pagination cursors

    for page_num in range(1, max_pages + 1):
        resp = epmc_get("search", params={
            "query":      query,
            "resultType": "core",       # "core" includes author list and MeSH terms
            "pageSize":   page_size,
            "cursorMark": cursor_mark,
            "format":     "json",
        })
        data = resp.json()

        # Unpack the results list (nested under resultList → result)
        results = data.get("resultList", {}).get("result", [])
        all_articles.extend(results)

        print(
            f"  Page {page_num}: +{len(results)} articles "
            f"(total so far: {len(all_articles)})",
            end="\r",
        )

        # Advance cursor; stop if unchanged (last page) or absent
        next_cursor = data.get("nextCursorMark")
        if not next_cursor or next_cursor == cursor_mark:
            break
        cursor_mark = next_cursor
        time.sleep(0.3)  # additional courtesy delay during bulk pagination

    print(f"\nDone. Total articles fetched: {len(all_articles)}")
    cache_path.write_text(json.dumps(all_articles))
    print(f"Cached to: {cache_path}")
    return all_articles


articles_raw = fetch_epmc_articles(SEARCH_QUERY, SEARCH_CACHE)
print(f"Records in memory: {len(articles_raw)}")

### 1.3 Parse Article Metadata into a Polars DataFrame

In [ ]:
def flatten_article(record: dict) -> dict:
    """
    Flatten a raw Europe PMC article record into a single row-friendly dict.

    Nested and list-valued fields (authors, MeSH terms, grant IDs) are
    collapsed to pipe-separated strings so each article occupies one row.

    Parameters
    ----------
    record : dict
        Raw article dict from the Europe PMC ``/search`` response.

    Returns
    -------
    dict
        Flat dict with the following keys:
        pmid, pmcid, doi, title, abstract, pub_year, journal_title,
        source, citation_count, is_open_access, authors, mesh_terms,
        grant_ids.
    """
    # ── Author list: "LastName Initials" for each author ─────────────────────
    author_list = record.get("authorList", {}).get("author", [])
    authors = " | ".join(
        f"{a.get('lastName', '')} {a.get('initials', '')}".strip()
        for a in author_list
        if a.get("lastName")
    )

    # ── MeSH descriptor names ─────────────────────────────────────────────────
    mesh_list = record.get("meshHeadingList", {}).get("meshHeading", [])
    mesh_terms = " | ".join(
        m.get("descriptorName", "") for m in mesh_list if m.get("descriptorName")
    )

    # ── Grant IDs ─────────────────────────────────────────────────────────────
    grant_list = record.get("grantsList", {}).get("grant", [])
    grant_ids = " | ".join(
        g.get("grantId", "") for g in grant_list if g.get("grantId")
    )

    return {
        "pmid":           record.get("pmid"),
        "pmcid":          record.get("pmcid"),
        "doi":            record.get("doi"),
        "title":          record.get("title"),
        "abstract":       record.get("abstractText"),
        "pub_year":       record.get("pubYear"),
        "journal_title":  record.get("journalTitle"),
        "source":         record.get("source"),
        "citation_count": record.get("citedByCount"),
        "is_open_access": record.get("isOpenAccess"),
        "authors":        authors or None,
        "mesh_terms":     mesh_terms or None,
        "grant_ids":      grant_ids or None,
    }


# ── Build the DataFrame ───────────────────────────────────────────────────────
rows = [flatten_article(r) for r in articles_raw]

articles_df = (
    pl.DataFrame(rows)
    .with_columns([
        # pub_year comes as a string from the API; cast to integer
        pl.col("pub_year").cast(pl.Int32, strict=False),
        # citation_count arrives as an integer already, but may be null
        pl.col("citation_count").cast(pl.Int32, strict=False),
        # Normalise open-access flag: "Y"/"N" → Boolean
        pl.col("is_open_access").eq("Y").alias("is_open_access"),
    ])
)

print(f"Shape  : {articles_df.shape}")
print(f"\nSchema :")
for col, dtype in articles_df.schema.items():
    print(f"  {col:<20} {dtype}")
print()
articles_df.head(10)